# fsociety — Fine-Tuning de Qwen2.5-Coder-1.5B
Fine-tuning con LoRA para programacion, exploiting y reversing.
---

In [ ]:
!pip install -qU transformers peft accelerate bitsandbytes datasets trl huggingface_hub

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='huggingface_hub')

import torch, os, json
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024

print("Cargando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer listo. Vocab: {tokenizer.vocab_size:,}")

In [ ]:
from datasets import load_dataset

print("Cargando dataset...")
ds = load_dataset('murdok1982/gemma4-programacion-seguridad', split='train', streaming=False)
print(f"Dataset: {len(ds):,} ejemplos")

def format_chatml(examples):
    texts = []
    for messages in examples['messages']:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {'text': texts}

ds = ds.map(format_chatml, batched=True, batch_size=1000)
ds = ds.train_test_split(test_size=0.05, seed=42)
train_ds = ds['train']
eval_ds = ds['test']
print(f"Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

gc.collect(); torch.cuda.empty_cache()
free = torch.cuda.mem_get_info()[0] / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM libre: {free:.1f} / {total:.0f} GiB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Cargando modelo en 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.config.use_cache = False

gc.collect(); torch.cuda.empty_cache()
used = torch.cuda.memory_allocated() / 1e9
print(f"VRAM usado: {used:.2f} GiB")
print(f"Parametros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT = "/content/fsociety-qwen"

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=100,
        num_train_epochs=1,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=1000,
        output_dir=OUTPUT,
        report_to="none",
        save_total_limit=2,
        load_best_model_at_end=True,
        push_to_hub=False,
    ),
)

print("Iniciando entrenamiento...")
trainer.train()

In [ ]:
from pathlib import Path
HF_USER = "murdok1982"
MODEL_NAME = "fsociety-LoRA"

model.save_pretrained(f"/content/{MODEL_NAME}")
tokenizer.save_pretrained(f"/content/{MODEL_NAME}")

if os.getenv("HF_TOKEN"):
    model.push_to_hub(f"{HF_USER}/{MODEL_NAME}")
    tokenizer.push_to_hub(f"{HF_USER}/{MODEL_NAME}")
    print(f"Subido a: {HF_USER}/{MODEL_NAME}")
else:
    print("HF_TOKEN no configurado, descarga local")

size = sum(f.stat().st_size for f in Path(f"/content/{MODEL_NAME}").rglob("*")) / 1e6
print(f"Tamaño adapters: {size:.1f} MB")

In [ ]:
from IPython.display import display, Markdown

messages = [
    {"role": "system", "content": "Eres fsociety, un experto en programacion y seguridad informatica."},
    {"role": "user", "content": "Escribe una funcion en Python que implemente busqueda binaria."},
]
test_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
display(Markdown(f"**Respuesta:**\n\n```python\n{response}\n```"))